# T2S Pipeline

Thin orchestrator: every cell just calls into a decoupled module and names its output via `results.py`. No modeling/training logic lives in this notebook — that all lives in the `.py` files, where it's unit-tested (`tests/`).

Expert policy pretraining is **skipped** — assumed already done, checkpoints under `train_res/expertPolicy/`.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import config
import results
config.ensure_dirs()
print('root:', config.ROOT_DIR)

## 1. Expert Policy Pretraining

Checkpoints live in `config.EXPERT_POLICY_DIR` = `train_res/expertPolicy/<config.EXPERT_RUN_NAME>/`, alongside `eval_history.json`. Toggle `RUN_EXPERT_TRAINING = False` (default) to just verify an existing run is there; `True` trains into that same folder.

In [ ]:
import expert_train, glob

RUN_EXPERT_TRAINING = False   # set True to actually train into config.EXPERT_POLICY_DIR

expert_run_dir = config.EXPERT_POLICY_DIR   # train_res/expertPolicy/<EXPERT_RUN_NAME>/
os.makedirs(expert_run_dir, exist_ok=True)

if RUN_EXPERT_TRAINING:
    results.new_run_dir('expert_policy', config.EXPERT_RUN_NAME, meta={'task_name': config.TASK_NAME})
    model, expert_history = expert_train.train_expert_policy(expert_run_dir)
    print('trained expert, final success rate:', expert_history[-1] if expert_history else None)
else:
    ckpts = glob.glob(os.path.join(expert_run_dir, f'{config.TASK_SLUG}_*.zip'))
    assert ckpts, (f'no expert checkpoints found in {expert_run_dir} for task {config.TASK_NAME} — '
                    'place them there (see chat for the rename command) or set RUN_EXPERT_TRAINING=True')
    print(f'reusing {len(ckpts)} expert checkpoints from {expert_run_dir}')

## 2. T2S Data Collection

In [ ]:
import data_collection
from env_utils import ensure_fixed_task

ensure_fixed_task()

DATA_RUN_NAME = 'v7'
data_run_dir = results.new_run_dir('data_collection', DATA_RUN_NAME,
                                     meta={'notes': 'coupling-based stage detection'})
plan = data_collection.default_collection_plan(expert_policy_dir=expert_run_dir,
                                                  include_noise_variants=True)
summary = data_collection.collect_dataset(data_run_dir, plan=plan)
print(summary)

## 3. T2S Model Training (MC / TD(0) / TD(&lambda;))

In [ ]:
import t2s_train

T2S_RUN_NAME = 'v1'
t2s_run_dir = results.new_run_dir('t2s_model', T2S_RUN_NAME,
                                    meta={'data_run': DATA_RUN_NAME})
dataset_path = os.path.join(data_run_dir, 'dataset.npz')

models, histories, summary_rows = t2s_train.run_all_combos(
    t2s_run_dir, dataset_path, seeds=(0,))
for row in summary_rows:
    print(row['combo'], 'mean val MSE:', round(row['mean_val_mse'], 1))

## 4. T2S Model Analysis (held-out policies — never in training data)

In [ ]:
import json
import t2s_predict, t2s_eval
from stable_baselines3 import SAC

EVAL_RUN_NAME = 'v1'
eval_run_dir = results.new_run_dir('t2s_eval', EVAL_RUN_NAME, meta={'t2s_run': T2S_RUN_NAME})

# TODO: point these at checkpoints that were genuinely never used in data_collection.default_collection_plan()
eval_success_pol = SAC.load('PATH/TO/held_out_success_checkpoint.zip')
eval_failure_pol = SAC.load('PATH/TO/held_out_failure_checkpoint.zip')

reports = {}
for combo in summary_rows:
    method, condition = combo['combo'].rsplit('_', 1)
    predict_fn = t2s_predict.load_t2s_predictor(t2s_run_dir, method, condition, seed=combo['best_seed'])
    reports[combo['combo']] = t2s_eval.run_full_evaluation(predict_fn, eval_success_pol, eval_failure_pol)

with open(os.path.join(eval_run_dir, 'eval_report.json'), 'w') as f:
    json.dump(reports, f, indent=2)
print(t2s_eval.format_report_table(reports))

t2s_eval.plot_combo_comparison(reports, save_path=os.path.join(eval_run_dir, 'combo_comparison.png'))

## 4b. Resume point — run this after a kernel restart

Rebuilds everything Section 5 needs (`t2s_run_dir`, `summary_rows`, `reports`) from files already on disk, so you can jump straight to downstream RL without re-running Stages 1-4. Run the first cell of the notebook (imports) first, then this, then Section 5.

In [ ]:
import json, os
import t2s_predict, t2s_eval, t2s_io

# --- point these at the runs you already completed ---
T2S_RUN_NAME  = 'v1'
EVAL_RUN_NAME = 'v1'

t2s_run_dir  = results.get_run_dir('t2s_model', T2S_RUN_NAME)
eval_run_dir = results.get_run_dir('t2s_eval', EVAL_RUN_NAME)

# rebuild summary_rows from the Stage 3 manifest (not recomputed — just read back)
manifest = t2s_io.load_manifest(t2s_run_dir)
summary_rows = [
    dict(combo=combo, mean_val_mse=info['mean_val_mse'], std_val_mse=info['std_val_mse'],
         seeds_val_mse=info['per_seed_val_mse'],
         mean_val_mse_succ_only=info['mean_val_mse_succ_only'],
         best_seed=info['best_seed'])
    for combo, info in manifest['combos'].items()
]

# rebuild reports from the Stage 4 eval report
with open(os.path.join(eval_run_dir, 'eval_report.json')) as f:
    reports = json.load(f)

print(f'resumed: {len(summary_rows)} combos from {t2s_run_dir}')
print(f'         {len(reports)} eval reports from {eval_run_dir}')
print()
print(t2s_eval.format_report_table(reports))

## 5. Downstream RL Training (SAC against the frozen T2S reward)

In [ ]:
import policy_train

# Select by Stage 4's held-out-POLICY generalization report (MAE on genuinely
# unseen behavior), NOT Stage 3's training-time val MSE (mean_val_mse in
# summary_rows) — the two can and do disagree (see chat: mc_succ won on val
# MSE but td0_succ generalized better on held-out policies).
SELECTION_METRIC = 'mae'   # or 'rmse' / swap sign and use 'spearman_rho'/'pearson_r' for a correlation-based pick

scored = {
    combo: r['success_scenario_summary'][SELECTION_METRIC]
    for combo, r in reports.items()
    if 'success_scenario_summary' in r and SELECTION_METRIC in r['success_scenario_summary']
}
assert scored, 'no combo has a usable success_scenario_summary — check Section 4 ran against real held-out policies'
CHOSEN_COMBO = min(scored, key=scored.get)
print('Stage 4 ranking (lower is better) by', SELECTION_METRIC + ':')
for combo, v in sorted(scored.items(), key=lambda kv: kv[1]):
    print(f'  {combo:14s} {v:.3f}')
print('chosen:', CHOSEN_COMBO)

method, condition = CHOSEN_COMBO.rsplit('_', 1)
best_seed = next(r['best_seed'] for r in summary_rows if r['combo'] == CHOSEN_COMBO)
predict_fn = t2s_predict.load_t2s_predictor(t2s_run_dir, method, condition, seed=best_seed)

POLICY_RUN_NAME = f"t2s_{CHOSEN_COMBO}_absolute_v1"
policy_run_dir = results.new_run_dir('policy', POLICY_RUN_NAME,
                                       meta={'t2s_run': T2S_RUN_NAME, 'combo': CHOSEN_COMBO,
                                             'selected_by': f'stage4_{SELECTION_METRIC}'})

model, history = policy_train.train_policy(policy_run_dir, predict_fn, reward_mode='absolute',
                                             total_timesteps=1_000_000)
print('final success rate:', history[-1] if history else 'no evals recorded')

## 5b. Inspect a single, manually-chosen policy

Load any checkpoint by name — an expert checkpoint, a mid-training snapshot, or a downstream policy — roll it out on the fixed scene, and look at what it actually does. Useful before committing to a long sweep.

Set `T2S_COMBO = None` to skip prediction overlay (e.g. when inspecting an expert policy on its own).

In [ ]:
import visualize, t2s_predict

# --- configure what to inspect ---
POLICY_DIR  = config.EXPERT_POLICY_DIR           # or: results.get_run_dir('policy', '<run_name>')
POLICY_CKPT = f'{config.TASK_SLUG}_final.zip'    # or 'policy_final.zip', 'policy_200000.zip', ...
T2S_COMBO   = 'td0_succ'                          # None to skip T2S prediction overlay
SEEDS_TO_VIEW = [0, 1, 2]
RENDER_VIDEO  = True

policy, resolved = visualize.load_policy(POLICY_CKPT, POLICY_DIR)
print('loaded:', resolved)

predict_fn = None
if T2S_COMBO:
    method, condition = T2S_COMBO.rsplit('_', 1)
    best_seed = next(r['best_seed'] for r in summary_rows if r['combo'] == T2S_COMBO)
    predict_fn = t2s_predict.load_t2s_predictor(t2s_run_dir, method, condition, seed=best_seed)
    print(f'T2S overlay: {T2S_COMBO} (seed {best_seed})')

inspect_dir = results.new_run_dir('policy', f'inspect_{os.path.basename(POLICY_DIR)}',
                                    meta={'checkpoint': resolved, 't2s_combo': T2S_COMBO})

rollouts = []
for sd in SEEDS_TO_VIEW:
    r = visualize.rollout(policy, predict_t2s=predict_fn, seed=sd, render=RENDER_VIDEO)
    rollouts.append(r)
    print(f'seed {sd}: {visualize.summarize_rollout(r)}')
    if RENDER_VIDEO and r['frames']:
        visualize.save_video(r['frames'], os.path.join(inspect_dir, f'rollout_seed{sd}.mp4'))

if predict_fn is not None:
    visualize.plot_rollouts(rollouts, labels=[f'seed {s}' for s in SEEDS_TO_VIEW],
                             save_path=os.path.join(inspect_dir, 't2s_curves.png'))
print(f'\nsaved to {inspect_dir}')

In [ ]:
# play one of the saved videos inline
from IPython.display import Video, display

video_path = os.path.join(inspect_dir, f'rollout_seed{SEEDS_TO_VIEW[0]}.mp4')
display(Video(video_path, embed=True, width=480)) if os.path.exists(video_path) else print('no video — set RENDER_VIDEO=True')

## 6. T2S model sweep for downstream RL (3 seeds each)

Trains a separate SAC policy for every (T2S combo x seed) pair. The scene is fixed, so the seed varies **SAC's** initialization, exploration noise and replay sampling — not the task. That's what tells you whether a T2S reward is *reliably* trainable rather than lucky once.

**This is expensive**: `len(COMBOS) x len(SEEDS)` full training runs. Start with a short `TIMESTEPS` to confirm the loop works end to end, then scale up.

In [ ]:
import json
import policy_train, t2s_predict

COMBOS    = ['mc_succ', 'td0_succ', 'tdlambda_succ']   # which T2S models to compare
SEEDS     = [0, 1, 2]
TIMESTEPS = 300_000        # start small to verify the loop; raise for real results
SWEEP_NAME = 'sweep_v1'

sweep_results = {}
for combo in COMBOS:
    method, condition = combo.rsplit('_', 1)
    best_seed = next(r['best_seed'] for r in summary_rows if r['combo'] == combo)
    # one frozen predictor per combo, reused across all SAC seeds so the only
    # thing varying within a combo is the RL seed
    predict_fn = t2s_predict.load_t2s_predictor(t2s_run_dir, method, condition, seed=best_seed)

    for sd in SEEDS:
        run_name = f'{SWEEP_NAME}_{combo}_seed{sd}'
        run_dir = results.new_run_dir('policy', run_name,
                                        meta={'sweep': SWEEP_NAME, 'combo': combo, 'rl_seed': sd,
                                              't2s_run': T2S_RUN_NAME, 't2s_seed': best_seed,
                                              'timesteps': TIMESTEPS})
        print(f'=== {combo} seed {sd} -> {run_name} ===')
        _model, history = policy_train.train_policy(
            run_dir, predict_fn, reward_mode='absolute',
            total_timesteps=TIMESTEPS, seed=sd)
        sweep_results[(combo, sd)] = history

sweep_rows = policy_train.summarize_sweep(sweep_results)
print()
print(policy_train.format_sweep_table(sweep_rows))

# persist so this survives a kernel restart (tuple keys -> strings for JSON)
sweep_dir = results.new_run_dir('policy', SWEEP_NAME, meta={'kind': 'sweep_summary'})
with open(os.path.join(sweep_dir, 'sweep_summary.json'), 'w') as f:
    json.dump({'rows': sweep_rows,
                'raw': {f'{c}|{s}': h for (c, s), h in sweep_results.items()}}, f, indent=2)
print(f'\nsaved to {sweep_dir}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# left: final success rate per combo, mean +/- std across seeds
combos = [r['combo'] for r in sweep_rows]
means  = [r['mean_final_success'] for r in sweep_rows]
stds   = [r['std_final_success'] for r in sweep_rows]
ax1.bar(combos, means, yerr=stds, capsize=5, color='tab:blue')
for i, r in enumerate(sweep_rows):   # individual seeds as dots, so outliers are visible
    ax1.scatter([i] * len(r['per_seed_final_success']), r['per_seed_final_success'],
                 color='black', zorder=3, s=25)
ax1.set_ylabel('final success rate'); ax1.set_ylim(0, 1)
ax1.set_title(f'Downstream RL success ({len(SEEDS)} seeds, bars = mean ± std)')
ax1.set_xticklabels(combos, rotation=20, ha='right')

# right: learning curves, one line per (combo, seed)
colors = dict(zip(COMBOS, ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']))
for (combo, sd), history in sweep_results.items():
    if not history:
        continue
    steps = [h['step'] for h in history]
    rates = [h['success_rate'] for h in history]
    ax2.plot(steps, rates, color=colors.get(combo, 'gray'), alpha=0.7,
              label=combo if sd == SEEDS[0] else None)
ax2.set_xlabel('environment steps'); ax2.set_ylabel('success rate'); ax2.set_ylim(0, 1)
ax2.set_title('Learning curves (one line per seed)'); ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(sweep_dir, 'sweep_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()